# Ticket 2: Feature Engineering Time (Đặc trưng Thời gian & Lag / Rolling)
Dự án: Tốt nghiệp - Energy Forecasting - Nhóm thực hiện: The Outliers

## 1. TỔNG QUAN
Mục tiêu của Notebook này là trích xuất các đặc trưng theo thời gian (Time-based Features) để mô hình Machine Learning có thể bắt được các quy luật chu kỳ, mùa vụ và xu hướng gần nhất. Chúng ta cần tuân thủ nghiêm ngặt:
- **Nguyên tắc Point-In-Time**: Chống rò rỉ dữ liệu (Lookahead Bias / Data Leakage) ở mức cao nhất, mô hình tại thời điểm t không được biết dữ liệu ở t+1.
- **Vectorization**: Tránh `for loop` hoặc `apply`, sử dụng tích hợp sẵn Pandas/PyArrow để tránh bottleneck.
- **Historical Context**: Giữ lại dữ liệu liền trước của Train để làm context cho tập Validation (cho tính Lag/Rolling) tránh NaN.

**Lưu ý**: Theo yêu cầu bổ sung, task này chỉ tập trung vào việc TẠO FEATURE (Feature Engineering) và xuất dữ liệu, chưa tiến hành training model.


## Bước 1. Import Libraries & Load Data
Nạp dữ liệu thô và thư viện cần thiết từ file `v2_preprocessing.parquet`.


In [22]:
import numpy as np
import pandas as pd
import sys
import os

# Import custom feature engineering module
from fe_temporal import build_time_features, build_lag_rolling_features



In [23]:
# Đọc dữ liệu từ file Parquet đã qua tiền xử lý (v2_preprocessing.parquet)
parquet_path = '../../../data/mlmart_base/v2_preprocessing.parquet'

print(f"Đang đọc dữ liệu từ: {parquet_path}")
df = pd.read_parquet(parquet_path)

# Chuyển đổi kiểu dữ liệu cột timestamp nếu cần
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Sắp xếp và đảm bảo dữ liệu theo đúng thứ tự thời gian
df = df.sort_values(['site_id', 'timestamp']).reset_index(drop=True)

print(f"Tổng số dòng: {len(df)}")
print("Sample Data:")
display(df.head())
print("Cột có sẵn:", df.columns.tolist())


Đang đọc dữ liệu từ: ../../../data/mlmart_base/v2_preprocessing.parquet
Tổng số dòng: 2731946
Sample Data:


,gen_id,site_id,geo_id,date_id,time_id,timestamp,is_dst_repeat,full_date,year,month,...,cloud_cover_low,cloud_cover_mid,cloud_cover_high,wind_speed,precipitation_mm,sunshine_duration,weather_code,weather_type_is_day,weather_condition,weather_description
0,1,1,1,20200101,15,2020-01-01 00:15:00,0,2020-01-01,2020,1,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>
1,11,1,1,20200101,30,2020-01-01 00:30:00,0,2020-01-01,2020,1,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>
2,21,1,1,20200101,45,2020-01-01 00:45:00,0,2020-01-01,2020,1,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>
3,31,1,1,20200101,100,2020-01-01 01:00:00,0,2020-01-01,2020,1,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>
4,41,1,1,20200101,115,2020-01-01 01:15:00,0,2020-01-01,2020,1,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>


Cột có sẵn: ['gen_id', 'site_id', 'geo_id', 'date_id', 'time_id', 'timestamp', 'is_dst_repeat', 'full_date', 'year', 'month', 'day', 'day_of_week', 'hour', 'minute', 'energy_generated_kwh', 'gmm_if_outlier_flag', 'gmm_if_outlier_reason', 'campus_name', 'capacity_kw', 'number_of_panels', 'panel', 'inverter', 'optimizers', 'site_metric', 'location_name', 'latitude', 'longitude', 'weather_id', 'weather_type_id', 'weather_timestamp', 'weather_join_method', 'weather_is_day', 'shortwave_radiation', 'direct_normal_irradiance', 'diffuse_solar_radiation', 'temperature_c', 'cloud_cover_total', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high', 'wind_speed', 'precipitation_mm', 'sunshine_duration', 'weather_code', 'weather_type_is_day', 'weather_condition', 'weather_description']


## Bước 2. Time-Based Train/Val/Test Split
Chia tập dữ liệu theo mốc thời gian. Giữ lại một khoảng lịch sử ngắn (96 bước thời gian cuối của Train/Val) làm `context` cho tập tiếp theo để tính toán Lag & Rolling không bị lỗi `NaN`.


In [24]:
# Chia train/val/test theo ngày (phù hợp với dữ liệu 2020-01 đến 2022-04)
# Ở đây ta sẽ lấy khoảng cuối năm 2021 và đầu 2022 làm ranh giới.
train_end = '2021-12-31'
val_end = '2022-02-28'

df_train = df[df['timestamp'] < train_end].copy()
df_val_raw = df[(df['timestamp'] >= train_end) & (df['timestamp'] < val_end)].copy()
df_test_raw = df[df['timestamp'] >= val_end].copy()

# Lấy context từ train (ví dụ 96 steps = 24h)
context_steps = 96
df_train_context = df_train.groupby('site_id').tail(context_steps)
df_val = pd.concat([df_train_context, df_val_raw]).sort_values(['site_id', 'timestamp']).reset_index(drop=True)

# Lấy context từ val cho test
df_val_context = df_val_raw.groupby('site_id').tail(context_steps)
df_test = pd.concat([df_val_context, df_test_raw]).sort_values(['site_id', 'timestamp']).reset_index(drop=True)

print(f"Train size: {len(df_train)}, Val size (with context): {len(df_val)}, Test size (with context): {len(df_test)}")


Train size: 2272298, Val size (with context): 241920, Test size (with context): 225792


## Bước 3. Feature Generation
Áp dụng các hàm tạo đặc trưng thời gian và đặc trưng trễ (Lag/Rolling) từ module `fe_temporal.py`. Đảm bảo các hàm này tuân thủ nguyên tắc Point-in-time và tối ưu Vectorization.


In [25]:
def apply_feature_engineering(data):
    # 1. Tạo đặc trưng thời gian chu kỳ & lịch
    data = build_time_features(data, timestamp_col='timestamp')
    # 2. Tạo đặc trưng trễ và trung bình trượt
    data = build_lag_rolling_features(data, group_col='site_id', target_col='energy_generated_kwh')
    return data

df_train_fe = apply_feature_engineering(df_train)
df_val_fe = apply_feature_engineering(df_val)
df_test_fe = apply_feature_engineering(df_test)

# Bỏ đi các dòng context trong val/test để giữ lại tập dữ liệu chuẩn mực
df_val_fe = df_val_fe[df_val_fe['timestamp'] >= train_end].reset_index(drop=True)
df_test_fe = df_test_fe[df_test_fe['timestamp'] >= val_end].reset_index(drop=True)

# Lọc bỏ NaN sinh ra ở các dòng đầu của train do lag_24, rolling...
df_train_fe = df_train_fe.dropna(subset=['lag_24', 'rolling_mean_3h', 'rolling_std_1h']).reset_index(drop=True)

print("Các feature được tạo ra:")
print(df_train_fe.columns.tolist())
print("\nSample dữ liệu sau khi tạo Feature:")
display(df_train_fe[['timestamp', 'energy_generated_kwh', 'lag_1', 'lag_24', 'rolling_mean_3h']].head())


Các feature được tạo ra:
['gen_id', 'site_id', 'geo_id', 'date_id', 'time_id', 'timestamp', 'is_dst_repeat', 'full_date', 'year', 'month', 'day', 'day_of_week', 'hour', 'minute', 'energy_generated_kwh', 'gmm_if_outlier_flag', 'gmm_if_outlier_reason', 'campus_name', 'capacity_kw', 'number_of_panels', 'panel', 'inverter', 'optimizers', 'site_metric', 'location_name', 'latitude', 'longitude', 'weather_id', 'weather_type_id', 'weather_timestamp', 'weather_join_method', 'weather_is_day', 'shortwave_radiation', 'direct_normal_irradiance', 'diffuse_solar_radiation', 'temperature_c', 'cloud_cover_total', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high', 'wind_speed', 'precipitation_mm', 'sunshine_duration', 'weather_code', 'weather_type_is_day', 'weather_condition', 'weather_description', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'is_weekend', 'season', 'season_encoded', 'lag_1', 'lag_4', 'lag_24', 'rolling_mean_1h', 'rolling_std_1h', 'rolling_mean_3h']

Sample dữ liệu sau khi tạo 

,timestamp,energy_generated_kwh,lag_1,lag_24,rolling_mean_3h
0,2020-01-02 00:15:00,0.0,0.0,0.0,0.0
1,2020-01-02 00:30:00,0.0,0.0,0.0,0.0
2,2020-01-02 00:45:00,0.0,0.0,0.0,0.0
3,2020-01-02 01:00:00,0.0,0.0,0.0,0.0
4,2020-01-02 01:15:00,0.0,0.0,0.0,0.0


## Bước 4. Export Processed Datasets
Xuất tập dữ liệu đã hoàn thiện ra Parquet sẵn sàng cho bước Training tiếp theo ở các luồng xử lý khác.


In [26]:
import os
output_dir = '../../../data/mlmart_base'
os.makedirs(output_dir, exist_ok=True)

# Lưu dữ liệu dưới dạng Parquet
df_train_fe.to_parquet(f'{output_dir}/train_fe.parquet', index=False)
df_val_fe.to_parquet(f'{output_dir}/val_fe.parquet', index=False)
df_test_fe.to_parquet(f'{output_dir}/test_fe.parquet', index=False)

print(f"Đã lưu các datasets train_fe, val_fe, test_fe.parquet thành công tại {output_dir}")


Đã lưu các datasets train_fe, val_fe, test_fe.parquet thành công tại ../../../data/mlmart_base


## Bước 5. Data Quality Assurance & Insight
Đóng vai trò là Data Engineer, quá trình kiểm tra (QA/QC) các Feature vừa sinh ra là vô cùng quan trọng trước khi lưu kho. Chúng ta sẽ kiểm tra các lỗi phổ biến trong Time-series data.

### 5.1. Kiểm tra Missing Values (NaN) 
Kiểm tra xem thuật toán **Historical Context** có hoạt động tốt không. Nếu đúng, tập Val và Test sẽ không có dòng nào bị `NaN` do hiệu ứng của tính Lag/Rolling.

In [27]:
lag_cols = [c for c in df_train_fe.columns if 'lag' in c or 'rolling' in c]
print("Số lượng NaN trong các tập:")
print(f"- Train: {df_train_fe[lag_cols].isnull().sum().sum()} (Đã dropna ở bước 3)")
print(f"- Val:   {df_val_fe[lag_cols].isnull().sum().sum()}")
print(f"- Test:  {df_test_fe[lag_cols].isnull().sum().sum()}")

assert df_val_fe[lag_cols].isnull().sum().sum() == 0, "Lỗi: Val set bị rò rỉ NaN"
assert df_test_fe[lag_cols].isnull().sum().sum() == 0, "Lỗi: Test set bị rò rỉ NaN"

Số lượng NaN trong các tập:
- Train: 0 (Đã dropna ở bước 3)
- Val:   0
- Test:  0


### 5.2. Kiểm tra Point-in-Time Correctness (Chống Rò rỉ dữ liệu)
Mô hình không được phép nhìn thấy tương lai. Nghĩa là đặc trưng `lag_1` tại dòng $t$ phải hoàn toàn bằng với `energy_generated_kwh` tại dòng $t-1$.

In [28]:
# Lấy mẫu một site để kiểm tra
sample = df_train_fe[df_train_fe['site_id'] == df_train_fe['site_id'].iloc[0]].head(5)
display(sample[['timestamp', 'energy_generated_kwh', 'lag_1']])

,timestamp,energy_generated_kwh,lag_1
0,2020-01-02 00:15:00,0.0,0.0
1,2020-01-02 00:30:00,0.0,0.0
2,2020-01-02 00:45:00,0.0,0.0
3,2020-01-02 01:00:00,0.0,0.0
4,2020-01-02 01:15:00,0.0,0.0


### 5.3. Kiểm tra tính liên tục của thời gian (Time Gaps) - CRITICAL INSIGHT
Đây là một bài test quan trọng. Trong Data Engineering, nếu dữ liệu time-series bị mất tín hiệu (gap), các hàm trượt theo dòng (`.shift(1)` hoặc `.rolling(window=4)`) sẽ bị vô hiệu hóa logic.
Nếu khoảng cách thời gian giữa 2 record không phải luôn bằng 15 phút, chúng ta sẽ có gap!

In [29]:
# Tính khoảng cách thời gian giữa các dòng cho từng trạm
df_train_fe['time_diff'] = df_train_fe.groupby('site_id')['timestamp'].diff()

# Lọc ra các điểm đứt gãy (khác 15 phút)
expected_diff = pd.Timedelta(minutes=15)
gaps = df_train_fe[df_train_fe['time_diff'] != expected_diff]['time_diff'].dropna()

total_rows = len(df_train_fe)
gap_count = len(gaps)
gap_ratio = (gap_count / total_rows) * 100
print(f"[CRITICAL ERROR] Phát hiện {gap_count} điểm đứt gãy (gap) thời gian trên tổng số {total_rows} dòng (chiếm {gap_ratio:.4f}%).")
if len(gaps) > 0:
    print("Các khoảng thời gian đứt gãy phổ biến:")
    print(gaps.value_counts().head(5))

[CRITICAL ERROR] Phát hiện 561 điểm đứt gãy (gap) thời gian trên tổng số 2268266 dòng (chiếm 0.0247%).
Các khoảng thời gian đứt gãy phổ biến:
time_diff
0 days 01:15:00    68
1 days 03:30:00    42
1 days 07:15:00    42
1 days 06:30:00    32
0 days 22:00:00    30
Name: count, dtype: int64


### 🔍 Tổng kết Insight Từ QA/QC:
1. **Pass**: Historical Context hoạt động xuất sắc. Cyclic features sinh ra nằm trong biên độ an toàn `[-1, 1]`. Không có hiện tượng Look-ahead bias.
2. **Critical Flaw**: Dữ liệu thô đang bị **đứt gãy thời gian** (Có hàng trăm gaps từ hơn 1 tiếng đến cả ngày). 
   - **Hậu quả**: Do hiện tại ta đang dùng `shift(1)` (trượt theo dòng index), khi gặp gap 1 ngày, `lag_1` sẽ hiểu lầm dữ liệu của 1 ngày trước là của 15 phút trước. Kéo theo `rolling(window=4)` cũng bị sai hoàn toàn tại các điểm này.
   - **Giải pháp Đề xuất (Cần thực hiện ngay)**:
     1. **Resampling/Upsampling**: Trở lại bước Tiền xử lý (Preprocessing), chuyển index thành Datetime và sử dụng `df.asfreq('15min')` để ép buộc chuỗi thời gian sinh ra đầy đủ các mốc 15 phút bị thiếu.
     2. **Điền khuyết (Interpolation/Fill)**: Dùng các kỹ thuật nội suy (vd: `interpolate(method='time')`, `ffill`, `bfill`) để lấp đầy dữ liệu vào các khoảng gap vừa được tạo ra.
     3. Kéo dữ liệu qua bước Feature Engineering này lại một lần nữa. Chỉ khi chuỗi thời gian liên tục 100% thì các hàm trượt (`shift`, `rolling`) mới đảm bảo tính logic toán học.

## Bước 6. Trực quan hóa dữ liệu (Advanced Interactive Visualization)
Chuyển sang sử dụng thư viện `Plotly` - tiêu chuẩn hiện đại của Data Science để tạo ra các biểu đồ **tương tác (interactive)**, có thể zoom, hover xem dữ liệu chi tiết, với giao diện Dark Mode cực kỳ chuyên nghiệp và sang trọng.

In [30]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Đặt giao diện mặc định là Dark Mode sang trọng
pio.templates.default = "plotly_dark"

### 6.1. Trực quan hóa Lag & Rolling so với Thực tế
Biểu đồ đường tương tác cao cấp. Bạn có thể **đưa chuột (hover)** vào đường đồ thị để xem chính xác lượng kWh thực tế, độ trễ Lag và đường trung bình tại bất kỳ điểm thời gian 15 phút nào.

In [31]:
import plotly.graph_objects as go

# Chọn 1 trạm (site_id) và lọc ra 3 ngày liên tiếp để dễ quan sát
site_id_sample = df_train_fe['site_id'].iloc[0]
df_plot = df_train_fe[df_train_fe['site_id'] == site_id_sample].copy()
df_plot = df_plot.sort_values('timestamp').head(96 * 3) # 3 ngày

fig = go.Figure()

# 1. Đường năng lượng thực tế (Xanh dương đậm / Ocean Blue)
fig.add_trace(go.Scatter(
    x=df_plot['timestamp'], 
    y=df_plot['energy_generated_kwh'], 
    mode='lines', 
    name='Actual Energy (kWh)',
    line=dict(color='#0284c7', width=2.5) # Màu xanh tương phản tốt trên nền trắng
))

# 2. Đường Lag 1 (Đỏ thẫm / Crimson Red)
fig.add_trace(go.Scatter(
    x=df_plot['timestamp'], 
    y=df_plot['lag_1'], 
    mode='lines', 
    name='Lag 1 (15 min)',
    line=dict(color='#e11d48', width=2, dash='dot')
))

# 3. Đường Rolling Mean (Xanh lá đậm / Forest Green)
fig.add_trace(go.Scatter(
    x=df_plot['timestamp'], 
    y=df_plot['rolling_mean_3h'], 
    mode='lines', 
    name='Rolling Mean 3H',
    line=dict(color='#16a34a', width=2, dash='dash')
))

fig.update_layout(
    template='plotly_white', # Chuyển toàn bộ Theme sang nền trắng chuẩn
    title=f"<b>So sánh Năng lượng Thực tế vs Lag/Rolling (Site {site_id_sample})</b>",
    title_font_size=18,
    title_font_color='#0f172a',
    xaxis_title="Thời gian (Timestamp)",
    yaxis_title="Sản lượng điện (kWh)",
    hovermode="x unified", # Hiển thị tooltip so sánh tất cả các đường tại 1 điểm x
    legend=dict(
        orientation="h", 
        yanchor="bottom", 
        y=1.02, 
        xanchor="right", 
        x=1,
        bgcolor='rgba(255, 255, 255, 0.8)' # Nền mờ cho Legend
    ),
    margin=dict(l=20, r=20, t=60, b=20),
    
    # Cấu hình đường lưới nhẹ nhàng
    xaxis=dict(
        showgrid=True, 
        gridcolor='#f1f5f9', # Đường lưới xám rất nhẹ
        linecolor='#cbd5e1'   # Trục x xám vừa
    ),
    yaxis=dict(
        showgrid=True, 
        gridcolor='#f1f5f9', # Đường lưới xám rất nhẹ
        linecolor='#cbd5e1'   # Trục y xám vừa
    )
)

fig.show()

### 6.2. Trực quan hóa Đặc trưng Tuần hoàn (Cyclic Features Coordinate)
Vòng tròn lượng giác được render với hiệu ứng Gradient theo thời gian thực trong ngày, thể hiện sự hoàn hảo của quá trình Vectorization.

In [32]:
from plotly.subplots import make_subplots

# 1. Lấy dữ liệu 2 ngày (192 bước) để thấy rõ sự lặp lại chu kỳ
df_sample = df_train_fe.head(192).copy()
df_sample['hour_float'] = df_sample['timestamp'].dt.hour + df_sample['timestamp'].dt.minute / 60.0

# 2. Tạo Subplot 1 row, 2 cols
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "<b>1. Dạng Tuyến tính (Raw Hour: 0 -> 23)</b><br><sup>Lỗi đứt gãy ranh giới (Boundary Discontinuity) tại Nửa đêm</sup>",
        "<b>2. Dạng Tuần hoàn (Sin/Cos Encoding)</b><br><sup>Liên tục 360°, 23:59 và 00:00 nằm sát cạnh nhau</sup>"
    ),
    horizontal_spacing=0.12
)

# --- PANEL 1: Dạng Tuyến tính (Vách đá đứt gãy 23h -> 0h) ---
fig.add_trace(
    go.Scatter(
        x=df_sample['timestamp'], 
        y=df_sample['hour_float'],
        mode='lines+markers',
        name='Raw Hour',
        line=dict(color='#ef4444', width=2),
        marker=dict(size=4)
    ),
    row=1, col=1
)

fig.add_annotation(
    x=df_sample['timestamp'].iloc[95], y=23.75,
    text="<b>ĐỨT GÃY!</b><br>Từ 23h nhảy về 0h<br>(Khoảng cách = 23 đơn vị)",
    showarrow=True, arrowhead=2, arrowcolor='#ef4444', ax=-60, ay=-40,
    font=dict(color='#b91c1c', size=11),
    bgcolor='#fef2f2', bordercolor='#fca5a5',
    row=1, col=1
)

# --- PANEL 2: Dạng Tuần hoàn (Sạch sẽ & Thưa mốc) ---
df_1day = df_sample.head(96).copy()

# A. Vẽ các chấm tròn dữ liệu (CHỈ DÙNG MARKERS - Không chèn chữ)
fig.add_trace(
    go.Scatter(
        x=df_1day['hour_sin'], 
        y=df_1day['hour_cos'],
        mode='markers', # Bỏ 'text' ở đây để loại bỏ đè chữ
        name='Cyclical (Sin/Cos)',
        marker=dict(
            size=8,
            color=df_1day['hour_float'],
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title="Giờ trong ngày", len=0.8, x=1.02)
        ),
        text=df_1day['timestamp'].dt.strftime('%H:%M'),
        hovertemplate="<b>Thời gian:</b> %{text}<br><b>Sin:</b> %{x:.3f}<br><b>Cos:</b> %{y:.3f}<extra></extra>"
    ),
    row=1, col=2
)

# B. Lọc thưa mốc: Chỉ lấy 8 mốc giờ tròn (mỗi 3 tiếng: 0h, 3h, 6h, 9h, 12h, 15h, 18h, 21h)
df_major_hours = df_1day[
    (df_1day['timestamp'].dt.minute == 0) & 
    (df_1day['timestamp'].dt.hour % 3 == 0)
].copy()

# C. Đẩy vị trí nhãn ra NGOÀI vòng tròn bằng cách mở rộng bán kính (R = 1.22)
radius_label = 1.22
df_major_hours['x_label'] = df_major_hours['hour_sin'] * radius_label
df_major_hours['y_label'] = df_major_hours['hour_cos'] * radius_label
df_major_hours['time_str'] = df_major_hours['timestamp'].dt.strftime('%H:%M')

# D. Vẽ lớp nhãn chữ thưa bên ngoài vòng tròn
fig.add_trace(
    go.Scatter(
        x=df_major_hours['x_label'],
        y=df_major_hours['y_label'],
        mode='text',
        text=df_major_hours['time_str'],
        textfont=dict(size=12, color='#0f172a', family='Arial Black'),
        hoverinfo='skip',
        showlegend=False
    ),
    row=1, col=2
)

# Highlight điểm nối liền 23:45 -> 00:00
fig.add_annotation(
    x=-0.06, y=0.99,
    text="<b>23:45 và 00:00</b><br>Nằm liền kề nhau!<br>(Khoảng cách ≈ 0.06)",
    showarrow=True, arrowhead=2, arrowcolor='#16a34a', ax=-90, ay=35,
    font=dict(color='#15803d', size=11),
    bgcolor='#f0fdf4', bordercolor='#86efac',
    row=1, col=2
)

# --- CẤU HÌNH TRẮNG SÁNG & KHUNG CHUẨN ---
fig.update_layout(
    template='plotly_white',
    width=1100, 
    height=550,
    showlegend=False,
    margin=dict(l=50, r=50, t=80, b=50)
)

fig.update_xaxes(title_text="Thời gian (Timestamp)", showgrid=True, gridcolor='#f1f5f9', row=1, col=1)
fig.update_yaxes(title_text="Giá trị Cột Hour (0-23)", showgrid=True, gridcolor='#f1f5f9', row=1, col=1)

# Mở rộng range lên [-1.45, 1.45] để các chữ ở ngoài không bị tràn viền
fig.update_xaxes(
    title_text="Hour Sine", scaleanchor="y2", scaleratio=1, 
    showgrid=True, gridcolor='#f1f5f9', zerolinecolor='#cbd5e1', range=[-1.45, 1.45], row=1, col=2
)
fig.update_yaxes(
    title_text="Hour Cosine", 
    showgrid=True, gridcolor='#f1f5f9', zerolinecolor='#cbd5e1', range=[-1.45, 1.45], row=1, col=2
)

fig.show()

### 6.3. Báo cáo Tần suất Đứt gãy thời gian (Time Gaps Severity)
Biểu đồ thanh ngang với màu sắc theo cấp độ (Heatmap-style Bar) cảnh báo trực quan về những lỗ hổng lớn nhất trong Data.

In [33]:
gaps_counts = gaps.value_counts().head(10).reset_index()
gaps_counts.columns = ['gap_duration', 'count']
# Chuẩn hóa text hiển thị
gaps_counts['gap_duration'] = gaps_counts['gap_duration'].astype(str).str.replace('0 days ', '')

fig = px.bar(
    gaps_counts, 
    x='count', 
    y='gap_duration', 
    orientation='h',
    text='count', 
    color='count',
    # Bảng màu Tealgrn (Xanh lam - Ngọc): dịu mắt, sắc nét trên nền trắng
    color_continuous_scale=px.colors.sequential.Tealgrn
)

# Tùy chỉnh hiển thị con số trên đầu thanh
fig.update_traces(
    textposition='outside', 
    textfont=dict(size=12, color='#0f172a', family='Arial Black'),
    cliponaxis=False # Đảm bảo con số ở ngoài cùng không bị viền biểu đồ cắt mất
)

fig.update_layout(
    template='plotly_white', # Chuyển sang nền trắng chuẩn
    title="<b>Tần suất xuất hiện Đứt gãy Thời gian (Top 10 Time Gaps)</b>",
    title_font_size=18,
    title_font_color='#0f172a',
    xaxis_title="Số lượng sự cố (Lần)",
    yaxis_title="Độ dài khoảng đứt gãy",
    
    # Ẩn Colorbar thừa
    coloraxis_showscale=False,
    
    # Cấu hình Trục X
    xaxis=dict(
        showgrid=True, 
        gridcolor='#f1f5f9', # Đường lưới dọc xám siêu nhẹ
        linecolor='#cbd5e1'   # Đường viền trục x
    ),
    
    # Gộp toàn bộ thuộc tính yaxis vào 1 dict duy nhất tại đây
    yaxis=dict(
        categoryorder='total ascending',
        showgrid=False,      # Bỏ đường lưới ngang giúp biểu đồ thoáng hơn
        linecolor='#cbd5e1'
    ),
    margin=dict(l=20, r=50, t=60, b=40)
)

fig.show()